# `polynomial.py` walkthrough

`data_generator_funs/polynomial.py` has three pieces:

1. `_draw_combinations` -- picks *which* parameter pairs/triplets get an interaction term, and marks a subset of them "sensitive"
2. `generate_matrix_full` -- assembles all the coefficients (linear, quadratic, [cubic], pairwise, [triplet]) using nested sensitivity
3. `compute_Y` -- turns X + those coefficients into Y

This notebook exercises each one directly with small, fully-printable numbers so you can see exactly what each does, rather than staring at a 20-param dataset.

In [1]:
import sys
from pathlib import Path

import numpy as np

sys.path.append(str(Path.cwd().parent))

from data_generator_funs import polynomial

## 1. `_draw_combinations` -- choosing which pairs/triplets exist, and which are sensitive

Signature: `_draw_combinations(rng, n_params, order, sensitive_pool, n_background, n_sensitive, sensitivity_multiplier, n_outputs)`

- `order=2` draws pairs, `order=3` draws triplets.
- It first draws `n_sensitive` combinations *only* from `sensitive_pool` (these get amplified).
- Then it draws `n_background` more combinations from whatever's left over (not amplified).
- Returns `(combos, coefficients)` with the sensitive combos listed first.

Below: 6 parameters, sensitive_pool = {1, 3, 5}. We ask for 1 sensitive pair (must be built from {1,3,5}) and 2 background pairs (from anywhere else).

In [ ]:
rng = np.random.default_rng(0)

combos, coeffs = polynomial._draw_combinations(
    rng, n_params=6, order=2,
    sensitive_pool=[1, 3, 5],
    n_background=2, n_sensitive=1,
    sensitivity_multiplier=10.0, n_outputs=1,
)

print("combos (sensitive first):", combos)
print("coefficients:\n", coeffs)
print()
print("sensitive pair", combos[0], "-> both indices in {1,3,5}:", set(combos[0]).issubset({1, 3, 5}))
print("background pairs are NOT required to avoid {1,3,5}, just not equal to the chosen sensitive pair")

combos (sensitive first): [(3, 5), (1, 4), (1, 5)]
coefficients:
 [[ 3.20211325]
 [ 0.10490012]
 [-0.53566937]]

sensitive pair (3, 5) -> both indices in {1,3,5}: True
background pairs are NOT required to avoid {1,3,5}, just not equal to the chosen sensitive pair


### Seeing the amplification directly

Run it twice with the same seed (so the same combos get drawn) but different `sensitivity_multiplier` -- only the sensitive row's coefficient should scale, the background rows stay identical.

In [11]:
coeffs_b

array([[ 3.20211325],
       [ 0.10490012],
       [-0.53566937]])

## 2. `generate_matrix_full` -- assembling the full coefficient set

This calls `_draw_combinations` internally (once for pairs, once more for triplets if `degree=3`) and also builds the dense per-parameter linear/quadratic/[cubic] coefficient matrices, amplifying the rows belonging to `paras_to_vary`.

Small example: 6 params, 2 outputs, 2 sensitive params, 2 background pairs + 1 sensitive pair.

In [5]:
result = polynomial.generate_matrix_full(
    n_params=6, n_outputs=2, degree=2,
    n_sensitive_para=2, sensitivity_multiplier=4.0,
    n_pairs=2, n_sensitive_pairs=1,
    seed=1,
)

for key, value in result.items():
    print(key, "=")
    print(value)
    print()

coefficients_linear =
[[ 0.34558419  0.82161814]
 [ 0.33043708 -1.30315723]
 [ 0.90535587  0.44637457]
 [-2.14781294  2.32447242]
 [ 0.3645724   0.2941325 ]
 [ 0.11368897  2.18685195]]

coefficients_quad =
[[-0.73645409 -0.16290995]
 [-0.48211931  0.59884621]
 [ 0.03972211 -0.29245675]
 [-3.12763385 -1.02876896]
 [ 0.00814218 -0.27560291]
 [ 5.17625526  4.02689726]]

pairs =
[(3, 5), (2, 3), (0, 5)]

coefficients_pairs =
[[-1.68876165  0.85457199]
 [ 0.21732193  2.11783876]
 [-1.11202076 -0.37760501]]

paras_to_vary =
[5, 3]



In [6]:
# Confirm the linear/quadratic rows for sensitive params really are ~4x a "no sensitivity" draw
baseline = polynomial.generate_matrix_full(
    n_params=6, n_outputs=2, degree=2,
    n_sensitive_para=0, n_pairs=0, seed=1,
)
sensitive_rows = result["paras_to_vary"]
print("sensitive params:", sensitive_rows)
print("coefficients_linear ratio (sensitive rows only):")
print(result["coefficients_linear"][sensitive_rows] / baseline["coefficients_linear"][sensitive_rows])
print("-> ~4.0 everywhere, matching sensitivity_multiplier")

sensitive params: [5, 3]
coefficients_linear ratio (sensitive rows only):
[[4. 4.]
 [4. 4.]]
-> ~4.0 everywhere, matching sensitivity_multiplier


In [6]:
# Confirm nesting: every sensitive pair is built only from sensitive params
n_sensitive_pairs = 1
sensitive_pairs = result["pairs"][:n_sensitive_pairs]
print("sensitive pairs:", sensitive_pairs)
print("all members in paras_to_vary:", all(set(p).issubset(set(sensitive_rows)) for p in sensitive_pairs))

sensitive pairs: [(3, 5)]
all members in paras_to_vary: True


## 3. `degree=3` -- adding cubic + triplet terms

Same idea, one order higher: `coefficients_cubic` is a dense (n_params, n_outputs) matrix like linear/quadratic, and `triplets`/`coefficients_triplets` work exactly like `pairs`/`coefficients_pairs` but with 3-tuples instead of 2-tuples, nested inside the same sensitive-parameter set.

In [7]:
result3 = polynomial.generate_matrix_full(
    n_params=6, n_outputs=2, degree=3,
    n_sensitive_para=3, sensitivity_multiplier=4.0,
    n_pairs=2, n_sensitive_pairs=1,
    n_triplets=1, n_sensitive_triplets=1,
    seed=2,
)

print("sensitive params:", result3["paras_to_vary"])
print("pairs (sensitive first):", result3["pairs"])
print("triplets (sensitive first):", result3["triplets"])
print()
sensitive_triplet = result3["triplets"][:1]
print("sensitive triplet built only from sensitive params:",
      all(set(t).issubset(set(result3["paras_to_vary"])) for t in sensitive_triplet))

sensitive params: [5, 2, 4]
pairs (sensitive first): [(4, 5), (1, 3), (0, 3)]
triplets (sensitive first): [(2, 4, 5), (2, 3, 4)]

sensitive triplet built only from sensitive params: True


## 4. `compute_Y` -- turning coefficients into data

`compute_Y(X, coefficients_linear, coefficients_quad, pairs, coefficients_pairs, coefficients_cubic=None, triplets=None, coefficients_triplets=None)` just evaluates:

```
y_k = sum_i b1[i,k]*x_i + sum_i b2[i,k]*x_i**2 + sum_(i,j) c2[(i,j),k]*x_i*x_j
    [+ sum_i b3[i,k]*x_i**3 + sum_(i,j,l) c3[(i,j,l),k]*x_i*x_j*x_l]
```

To make sure that's really all it does, we recompute output column 0 by hand with plain Python loops (no matrix ops) and check it matches `compute_Y`'s vectorized result.

In [8]:
rng = np.random.default_rng(42)
X = rng.normal(size=(5, 6))  # 5 samples, 6 params

Y = polynomial.compute_Y(
    X, result["coefficients_linear"], result["coefficients_quad"],
    result["pairs"], result["coefficients_pairs"],
)

# Hand-rolled recomputation of column 0, one sample at a time
k = 0
Y_manual = []
for row in X:
    total = 0.0
    for i in range(6):
        total += result["coefficients_linear"][i, k] * row[i]
        total += result["coefficients_quad"][i, k] * row[i] ** 2
    for (i, j), c in zip(result["pairs"], result["coefficients_pairs"][:, k]):
        total += c * row[i] * row[j]
    Y_manual.append(total)

print("compute_Y column 0:      ", Y[:, k])
print("hand-rolled column 0:    ", np.array(Y_manual))
print("match:", np.allclose(Y[:, k], Y_manual))

compute_Y column 0:       [ 5.69846558  3.98025561  3.12512337  0.12842514 -0.02550622]
hand-rolled column 0:     [ 5.69846558  3.98025561  3.12512337  0.12842514 -0.02550622]
match: True


In [9]:
# Same check for degree=3 (adds the cubic + triplet terms)
X3 = rng.normal(size=(5, 6))

Y3 = polynomial.compute_Y(
    X3, result3["coefficients_linear"], result3["coefficients_quad"],
    result3["pairs"], result3["coefficients_pairs"],
    coefficients_cubic=result3["coefficients_cubic"],
    triplets=result3["triplets"], coefficients_triplets=result3["coefficients_triplets"],
)

k = 0
Y3_manual = []
for row in X3:
    total = 0.0
    for i in range(6):
        total += result3["coefficients_linear"][i, k] * row[i]
        total += result3["coefficients_quad"][i, k] * row[i] ** 2
        total += result3["coefficients_cubic"][i, k] * row[i] ** 3
    for (i, j), c in zip(result3["pairs"], result3["coefficients_pairs"][:, k]):
        total += c * row[i] * row[j]
    for (i, j, l), c in zip(result3["triplets"], result3["coefficients_triplets"][:, k]):
        total += c * row[i] * row[j] * row[l]
    Y3_manual.append(total)

print("compute_Y column 0:      ", Y3[:, k])
print("hand-rolled column 0:    ", np.array(Y3_manual))
print("match:", np.allclose(Y3[:, k], Y3_manual))

compute_Y column 0:       [-1.55946897  7.25146726  9.47737613 -3.0524986   1.26562191]
hand-rolled column 0:     [-1.55946897  7.25146726  9.47737613 -3.0524986   1.26562191]
match: True


## Recap: what each argument controls

| Argument | Controls |
|---|---|
| `degree` | 2 = linear+quadratic+pairs; 3 = also cubic+triplets |
| `n_sensitive_para` | how many individual params get amplified linear/quad/[cubic] coefficients |
| `sensitivity_multiplier` | the amplification factor applied to every sensitive element (param, pair, or triplet) |
| `n_pairs` | # of background (non-amplified) pairwise interaction terms |
| `n_sensitive_pairs` | # of pairwise terms drawn *from the sensitive params only*, and amplified |
| `n_triplets` | # of background triplet interaction terms (degree=3 only) |
| `n_sensitive_triplets` | # of triplet terms drawn from the sensitive params only, and amplified (degree=3 only) |

Total pairs in the model = `n_pairs + n_sensitive_pairs` (same for triplets).